In [1]:
import os
from cuml.cluster import KMeans
import numpy as np
import torch
import pandas as pd
from utils import utils
from utils.utils import load_tensor

In [2]:
N_CLUSTERS = 42

METADATA_JSON_FILE = "embeddings/id_metadata.json"
TENSOR_DIRECTORY = "embeddings/uuid_embeddings/embeddings/"

In [ ]:

# loads uuid -> tensor (non-pooled, [N, D])
tensors = load_tensor(TENSOR_DIRECTORY, num_workers=16)


Loading cache from embeddings/cache


In [ ]:
import json
# AlbumID, AlbumName, TrackID, Artists (Json [{"id": "artist_id", "name": "artist_name"}])
with open(METADATA_JSON_FILE, "r", encoding="utf-8") as f:
  metadata_dict = json.load(f)

metadata_df = pd.DataFrame(metadata_dict)

metadata_df.head()


,AlbumID,AlbumName,TrackID,Artists
0,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,1c2377e8-d3f1-4a8f-82c9-1fbede9de0d0,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...
1,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,5c27d4ff-cf85-42fc-aa51-c9856be198b3,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...
2,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,7ecfe8f0-9737-4fd5-9eaf-96770bd4226f,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...
3,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,bd12b3ff-0b53-44da-b615-118ce11b6c22,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...
4,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,c54e88c1-076e-4cc7-bce9-d861ada89ed2,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...


In [5]:
# metadata_df
# turn df into dict index with TrackID
metadata_dict = metadata_df.set_index("TrackID").to_dict(orient="index")


In [6]:
metadata_dict

{'1c2377e8-d3f1-4a8f-82c9-1fbede9de0d0': {'AlbumID': '00016005-985e-4f11-b06d-e2f37fd65608',
  'AlbumName': 'Taste me up!',
  'Artists': [{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae',
    'name': 'DiGiTAL WiNG'}]},
 '5c27d4ff-cf85-42fc-aa51-c9856be198b3': {'AlbumID': '00016005-985e-4f11-b06d-e2f37fd65608',
  'AlbumName': 'Taste me up!',
  'Artists': [{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae',
    'name': 'DiGiTAL WiNG'}]},
 '7ecfe8f0-9737-4fd5-9eaf-96770bd4226f': {'AlbumID': '00016005-985e-4f11-b06d-e2f37fd65608',
  'AlbumName': 'Taste me up!',
  'Artists': [{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae',
    'name': 'DiGiTAL WiNG'}]},
 'bd12b3ff-0b53-44da-b615-118ce11b6c22': {'AlbumID': '00016005-985e-4f11-b06d-e2f37fd65608',
  'AlbumName': 'Taste me up!',
  'Artists': [{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae',
    'name': 'DiGiTAL WiNG'}]},
 'c54e88c1-076e-4cc7-bce9-d861ada89ed2': {'AlbumID': '00016005-985e-4f11-b06d-e2f37fd65608',
  'AlbumName': 'Taste me up!',
  'Artists':

In [ ]:
# compute index of tensor to track id
from typing import Dict

missing_keys: Dict[str, int] = {}
for idx, key in enumerate(tensors.keys()):
  uuid = key.split(".")[0]
  try:
    metadata_dict[uuid]["TensorIdx"] = idx
  except:
    missing_keys[uuid] = idx

print(f"Missing keys: {missing_keys}, diff: {len(missing_keys)}")


Missing keys: {'0111e11f-2e2b-4ff1-8d7c-be6c45ea1b2c': 586, '015b8712-c2de-4a8f-baa0-b8b5d02155e3': 732, '01e86b65-6348-4bc9-af30-fd580d905919': 1051, '0320e638-0f54-41f6-8b6f-1ea22b160ddc': 1689, '038b889d-9bd4-4d7d-ba11-95ac87b9ef86': 1899, '041e2616-6b48-4548-b4c5-c216953ca998': 2195, '0436a26a-6c3a-4100-afee-e028ec246e3c': 2247, '04b688bf-0327-4235-8f5a-bcaf3c9defbd': 2517, '0578de3d-1e2d-4368-9480-806c21ceeb47': 2937, '06012c8f-8f50-41b2-9a35-858b66db8b14': 3237, '063e2725-8d30-4496-b1f0-acd84da40c15': 3372, '07c3cb30-654f-4269-a005-de1087d34d40': 4198, '0916c1bc-42e6-4c48-84f6-8cdc68cac18c': 4931, '091f98ca-5948-4102-9018-64e48c8bc25c': 4950, '0a32c882-c217-4872-b2ec-cd8a5b12ec78': 5503, '0a5027ab-45e3-4f11-a08e-f17c12c1aac1': 5562, '0cc6e0a2-7365-4151-8a04-bae20f553df4': 6866, '0d8dd2dd-3b22-4435-947e-b3624c33ca07': 7294, '0e2072b0-a591-484d-a306-40ddc0bc46ff': 7598, '100bbf91-3f2a-4acb-86fc-6de39d1e7089': 8615, '119b2540-9268-45ce-bce1-c9300ba24130': 9438, '12227217-6cb3-421b-8

In [8]:
emb_idx_df = pd.DataFrame.from_dict(metadata_dict, orient='index')
emb_idx_df.dropna(inplace=True)
emb_idx_df['TensorIdx'] = emb_idx_df['TensorIdx'].astype(int)
emb_idx_df = emb_idx_df.reset_index().rename(columns={'index': 'TrackID'})

emb_idx_df.head()

,TrackID,AlbumID,AlbumName,Artists,TensorIdx
0,1c2377e8-d3f1-4a8f-82c9-1fbede9de0d0,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...,15228
1,5c27d4ff-cf85-42fc-aa51-c9856be198b3,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...,49801
2,7ecfe8f0-9737-4fd5-9eaf-96770bd4226f,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...,68445
3,bd12b3ff-0b53-44da-b615-118ce11b6c22,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...,102141
4,c54e88c1-076e-4cc7-bce9-d861ada89ed2,00016005-985e-4f11-b06d-e2f37fd65608,Taste me up!,[{'id': 'e95a42e9-10c8-4b24-b88f-72d4626ac4ae'...,106697


In [9]:
emb_idx_df['ArtistIDList'] = emb_idx_df['Artists'].apply(lambda x: [artist['id'] for artist in x])

track_to_artists_map = pd.Series(emb_idx_df.ArtistIDList.values, index=emb_idx_df.TrackID).to_dict()
track_to_artists_map

{'1c2377e8-d3f1-4a8f-82c9-1fbede9de0d0': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 '5c27d4ff-cf85-42fc-aa51-c9856be198b3': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 '7ecfe8f0-9737-4fd5-9eaf-96770bd4226f': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'bd12b3ff-0b53-44da-b615-118ce11b6c22': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'c54e88c1-076e-4cc7-bce9-d861ada89ed2': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'cf588af1-8ad4-4ea4-9ca7-466a14643d92': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'cf72b9a9-292f-451b-8f03-c3e54304b6ea': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'd173f51d-fa33-4c09-aeed-91c468d5b208': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'd33ff253-f4f0-4f2e-80c6-c6f06fbb6ca1': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'f4808ffc-979c-42ea-90a4-462e82fb47eb': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 'fadcd744-aba8-47e1-8c2c-6e3da7e6c5f9': ['e95a42e9-10c8-4b24-b88f-72d4626ac4ae'],
 '558d16f1-dea1-461e-a000-3edd2308d377': ['6cd927f5-808e-48b9-85db-1625a0e35114'],
 '5b

In [ ]:
pooling_policies = ["mean", "mean+max"]
# pooled embeddings
all_embeddings = utils.pool_loaded_tensor_dict(
  tensors=tensors, mode='mean+max'
)

Detected allchunks tensors, pooling to 1D embedding (mean+max)


100%|██████████| 138748/138748 [00:02<00:00, 46297.66it/s]


In [ ]:
from typing import Dict, List
from collections import defaultdict

# compute centroid for each artist
artist_to_track_indicies: Dict[str, List[int]] = defaultdict(list)

track_id_to_idx = emb_idx_df.set_index("TrackID").to_dict(orient="index")

for track_id, metadata in track_id_to_idx.items():
  for artist in metadata['Artists']:
    artist_to_track_indicies[artist['id']].append(metadata['TensorIdx'])

embedding_mat = np.array(list(all_embeddings.values()))
artist_centroid: Dict[str, np.ndarray] = {}
for artist, track_indicies in artist_to_track_indicies.items():
  track_embeds = embedding_mat[np.array(track_indicies)]
  artist_centroid[artist] = track_embeds.mean(axis=0)

artist_centroid


{'e95a42e9-10c8-4b24-b88f-72d4626ac4ae': array([ 0.01052091,  0.01198753, -0.00153312, ...,  0.00815917,
         0.01135257,  0.01014479], dtype=float32),
 '6cd927f5-808e-48b9-85db-1625a0e35114': array([0.0082783 , 0.01477114, 0.00152859, ..., 0.00838992, 0.01047316,
        0.01220383], dtype=float32),
 '7725a3ee-3a92-4eb6-b5cd-b5461d402e89': array([ 0.01017172,  0.01261758, -0.00069952, ...,  0.00767819,
         0.0109442 ,  0.01135653], dtype=float32),
 'faa689b4-b345-4e0d-be19-066eebf55a56': array([ 0.00837497,  0.01492363, -0.00130169, ...,  0.00567051,
         0.01068038,  0.00913306], dtype=float32),
 'a6065265-6413-4541-bc7e-e7f466666dd0': array([0.00923053, 0.01318991, 0.00108577, ..., 0.0073954 , 0.01024916,
        0.0118218 ], dtype=float32),
 '2d047b2b-90d0-4b22-aa59-2f0d64c3bd6a': array([0.00857806, 0.01262907, 0.00178784, ..., 0.0073542 , 0.01143436,
        0.01172571], dtype=float32),
 '5889af12-024d-4aa8-8471-c3897643c1a1': array([0.00885605, 0.0137177 , 0.00225191

In [ ]:
# we want all our artist to have a vector representation
def knn_search(vectors: Dict[str, np.array], query_vector: np.array, k: int) -> Dict[str, List[str]]:
  """
  KNN search for the k nearest neighbors of the query vector in the given L2 normalized vectors.
  """

  agg_vectors = np.array(list(vectors.values()))

  # compute pairwise distances
  distances = np.linalg.norm(agg_vectors - query_vector, axis=1)

  # get top k indices
  top_k_indices = np.argsort(distances)[:k]
  
  reg_top_k_indices = [k.item() for k in top_k_indices]

  # get top k artists
  top_k_artists = [list(vectors.keys())[i] for i in reg_top_k_indices]

  return top_k_artists

def get_artist_name(artist_id: str) -> str:
  return emb_idx_df[emb_idx_df['ArtistIDList'].apply(lambda x: artist_id in x)]['Artists'].iloc[0][0]['name']

artist_id = '85de2a90-ea44-473f-af2f-2c22dcc99064'
print(f"Searching 10 nearest neighbors for {artist_id} (Centroid)")
result_ids = knn_search(artist_centroid, artist_centroid[artist_id], 10)
for result_id in result_ids:
  print(get_artist_name(result_id))
  

Searching 10 nearest neighbors for fdf7d27c-f263-4739-96b9-67567fe47df7 (Centroid)
森羅万象
ふぉれすとぴれお
pikapika
株式会社虎の穴
SOUND HOLIC
Amorevole
紺碧studio
セブンスヘブンMAXION
博丽神社事务所
COOL&CREATE
